In [0]:
dbutils.widgets.text("catalog","fk")
catalog = dbutils.widgets.get("catalog")

#stores

In [0]:
#sql
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.bronze.stores AS
SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
 FROM read_files('/Volumes/{catalog}/landing/raw/stores/stores.csv')
""")

In [0]:
#python
# from pyspark.sql.functions import *;

# store_df = spark.read \
#     .format("csv") \
#     .option("header","true") \
#     .load(f"/Volumes/{catalog}/landing/raw/stores/stores.csv")

# store_df = store_df.withColumn("_source_file",col("_metadata.file_path")) \
#     .withColumn("_ingested_at", current_timestamp())

# store_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema","true") \
#     .saveAsTable(f"{catalog}.bronze.stores")

#customers

In [0]:
spark.sql(f"""CREATE TABLE if not exists {catalog}.bronze.customers""")

In [0]:
spark.sql(f"""
COPY INTO {catalog}.bronze.customers
FROM (
    SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
    FROM '/Volumes/{catalog}/landing/raw/customers/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')  
COPY_OPTIONS ('mergeSchema' = 'true')        
""")

#products

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.bronze.products COMMENT "Products in the store" """)

In [0]:
spark.sql(f"""
  COPY INTO {catalog}.bronze.products
  FROM (
      SELECT *,
      _metadata.file_path AS _source_file,
      current_timestamp() AS _ingested_at
      FROM '/Volumes/{catalog}/landing/raw/products/'
  )
  FILEFORMAT = JSON
  FORMAT_OPTIONS ('multiLine' = 'true', 'inferSchema' = 'false')  
  COPY_OPTIONS (
    'mergeSchema'='true'
  )        
""")

#orders

In [0]:
from pyspark.sql.functions import *

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation","/Volumes/fk/ops/pipeline_state/orders_schema/") \
    .option("cloudFiles.schemaEvolutionMode","addNewColumns") \
    .option("multiline","true") \
    .option("cloudFiles.inferColumnTypes","true") \
    .load(f"/Volumes/{catalog}/landing/raw/orders/")

df = df.withColumn("_source_file",col("_metadata.file_path")) \
    .withColumn("_ingested_at", current_timestamp())

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/orders_checkpoint") \
    .trigger(availableNow=True) \
    .toTable(f"{catalog}.bronze.orders")